In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
class PotatoDataset(Dataset):


    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.transform = transform

        # Get all class names (folder names)
        self.classes = sorted(os.listdir(root_dir))

        # Create a mapping from class name to integer label
        self.class_to_idx = {class_name: i for i, class_name in enumerate(self.classes)}

        # Collect all image paths and their labels
        self.image_paths = []
        self.labels = []

        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            class_idx = self.class_to_idx[class_name]

            # Get all image files in this class folder
            for img_name in os.listdir(class_dir):
                img_path = os.path.join(class_dir, img_name)
                self.image_paths.append(img_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        # Load image
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')  # Ensure RGB format

        # Get label
        label = self.labels[idx]

        # Apply transforms if provided
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dir = os.path.join(path, "PlantVillage/train")
test_dir = os.path.join(path, "PlantVillage/test")

train_dataset = PotatoDataset(root_dir=train_dir, transform=transform)
test_dataset = PotatoDataset(root_dir=test_dir, transform=transform)

In [ ]:
import random
import numpy as np

# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):

    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import torch
# Visualize training samples
visualize_samples(train_dataset, num_samples=8, title="Training Dataset Samples")

In [ ]:
# Write your code here
from torchvision import models
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import kagglehub
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load pretrained EfficientNet-B0
model = models.efficientnet_b0(pretrained=True)

# Modify the classifier for binary classification (cats vs. dogs)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)

In [ ]:
# Write your code here
from tqdm import tqdm

# Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Track accuracy
        predictions = torch.sigmoid(outputs) > 0.5
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute accuracy
            predictions = torch.sigmoid(outputs) > 0.5
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


In [ ]:

from torchvision import transforms
from torch.utils.data import DataLoader



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)


print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")


In [ ]:
# Write your code here
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)
num_epochs = 5


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, valid_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here

class Residual_Connection(nn.Modlue):
      def __init__(self, num_classes=10):
        super(Residual_Connection, self).__init__()
        # Load pre-trained VGG16 model
        rc = models.resnet(weights=models.ResNet152_Weights.DEFAULT)

        # Freeze all parameters in the feature extractor
        for param in rc.features.parameters():
            param.requires_grad = False

        self.features = rc.features
        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 1, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

      def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

classes = ["Potato___Early_blight", "Potato___Late_blight", "Potato___healthy"]
model = Residual_Connection(num_classes=len(classes)).to(device)
print(model)
